# ALS: вес статуса planned

Меняем **только вес planned: 0,3 / 0,5 / 1,0**, при **score_coef=0**,
выбранном в предыдущем опыте. Для каждого веса используем seed 42, 7, 2026.
Остальные веса: completed=1, watching=1, on_hold=0,5, dropped=0.
Оценки ниже 5 исключаются; затухание выключено.

Сохраняем временные границы, фильтр 5/5 только на train и полную train-history
для исключений. Цели тех же 7 184 warm-validation пользователей не фильтруются
по доступности модели. ALS: factors=64, regularization=0.1, alpha=20,
iterations=20, num_threads=4; confidence=1+alpha*weight.

Сравниваем средний NDCG@10, разброс seed и парные разницы с planned=0,5,
а также precision, recall и покрытие каталога. Seed характеризуют чувствительность
к инициализации на одном validation, а не независимые выборки.
Test не оцениваем. Последний период уже изучался в старом EDA, поэтому его
независимость ограничена. Defaults и исходные артефакты не меняем.

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import sys
import json
from pathlib import Path
from time import perf_counter
import importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path.cwd().parent))
from app.training.datasets import load_anilist, filter_interactions
from app.training.evaluate import chronological_split, relevant_items, evaluate, ndcg_at_k
from app.training.weights import weighted_interactions, DEFAULT_STATUS_WEIGHTS
from app.recsys.popularity import PopularityRecommender
from app.recsys.als import ALSRecommender

pd.set_option("display.float_format", "{:,.6f}".format)
VALIDATION_START = pd.Timestamp("2026-09-02T00:34:23Z")
TEST_START = pd.Timestamp("2026-09-30T00:34:23Z")
SNAPSHOT_END = pd.Timestamp("2026-10-28T00:34:23Z")
PLANNED_WEIGHTS = [0.3, 0.5, 1.0]
SCORE_COEF = 0.0
BASE_STATUS_WEIGHTS = DEFAULT_STATUS_WEIGHTS.copy()
assert BASE_STATUS_WEIGHTS == dict(completed=1.0, watching=1.0, on_hold=0.5, planned=0.5, dropped=0.0)
SEEDS = [42, 7, 2026]
ALS_PARAMS = dict(factors=64, regularization=0.1, alpha=20, iterations=20, num_threads=4)
versions = {name: importlib.metadata.version(name) for name in ["implicit", "numpy", "scipy", "pandas"]}
print(versions)
inter = load_anilist()
assert inter.updated_at.max() == SNAPSHOT_END, "Снимок изменился: пересмотрите протокол"
history, validation, _ = chronological_split(
    inter, validation_start=VALIDATION_START, test_start=TEST_START)
training = filter_interactions(history, min_user_items=5, min_item_users=5)
counts = history.user_id.value_counts()
targets = {u: items for u, items in relevant_items(validation).items() if counts.get(u, 0) >= 5}
users = sorted(targets)
seen = history.groupby("user_id").anime_id.agg(set).to_dict()
print(f"Train после фильтра: {len(training):,} записей; warm-validation: {len(users):,} пользователей")
assert len(users) == 7184
assert len(history) == 3513393 and len(training) == 3499908
assert len(validation) == 218925

/home/deonik3/AniFolder/ml/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'implicit': '0.7.3', 'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0'}


Train после фильтра: 3,499,908 записей; warm-validation: 7,184 пользователей


## Контрольный baseline

В этой серии меняется только ALS. Популярность за 28 дней должна воспроизводить
предыдущий результат ≈ 0,113 NDCG@10.

In [2]:
popular = PopularityRecommender(window_days=28).fit(training, now=VALIDATION_START, history=history)
popular_metrics = evaluate(popular.recommend_many(users, k=10), targets)
assert np.isclose(popular_metrics["ndcg@10"], 0.113021, atol=5e-7, rtol=0)
pd.Series(popular_metrics, name="популярность 28 дней").to_frame()

,популярность 28 дней
precision@10,0.099624
recall@10,0.064522
ndcg@10,0.113021
coverage@10,65.000000
users,"7,184.000000"


## Девять запусков

В каждом варианте копируем DEFAULT_STATUS_WEIGHTS и меняем только planned.
Проверяем одинаковый набор положительных пар и неизменность остальных весов.
Полная история исключений остаётся прежней. Fallback не добавляем: пустая
выдача учитывается как нулевое качество.

In [3]:
records = []
per_user = {}
reference_pairs = None
reference_other_weights = None
for planned in PLANNED_WEIGHTS:
    status_weights = BASE_STATUS_WEIGHTS.copy()
    status_weights["planned"] = planned
    weighted = weighted_interactions(training, status_weights=status_weights,
                                     score_coef=SCORE_COEF, min_score=5, half_life_days=None)
    assert DEFAULT_STATUS_WEIGHTS == BASE_STATUS_WEIGHTS
    assert len(weighted) == 3293589
    assert not weighted.status.eq("dropped").any()
    assert not weighted.score.lt(5).fillna(False).any()
    np.testing.assert_allclose(weighted.loc[weighted.status.eq("planned"), "weight"], planned)
    other_weights = weighted.loc[weighted.status.ne("planned"), "weight"]
    if reference_other_weights is None:
        reference_other_weights = other_weights.copy()
    else:
        pd.testing.assert_series_equal(other_weights, reference_other_weights)
    pairs = weighted[["user_id", "anime_id"]]
    if reference_pairs is None:
        reference_pairs = pairs.copy()
    else:
        pd.testing.assert_frame_equal(pairs, reference_pairs)
    for seed in SEEDS:
        started = perf_counter()
        model = ALSRecommender(**ALS_PARAMS, random_state=seed).fit(weighted, history=history)
        fit_seconds = perf_counter() - started
        started = perf_counter()
        recs = model.recommend_many(users, k=10)
        recommend_seconds = perf_counter() - started
        assert all(len(r) == len(set(r)) and not (set(r) & seen.get(u, set())) for u, r in recs.items())
        assert set(recs) == set(users)
        metrics = evaluate(recs, targets)
        per_user[(planned, seed)] = np.array([ndcg_at_k(recs[u], targets[u], 10) for u in users])
        records.append({"planned_weight": planned, "seed": seed, **metrics,
                        "training_rows": len(weighted),
                        "short_lists": sum(len(r) < 10 for r in recs.values()),
                        "fit_seconds": fit_seconds, "recommend_seconds": recommend_seconds})
        print(f"planned={planned}, seed={seed}: NDCG@10={metrics['ndcg@10']:.6f}, "
              f"coverage={metrics['coverage@10']}, train={fit_seconds:.1f}s", flush=True)
results = pd.DataFrame(records)
display(results.set_index(["planned_weight", "seed"]))

planned=0.3, seed=42: NDCG@10=0.099330, coverage=4077, train=7.0s


planned=0.3, seed=7: NDCG@10=0.098172, coverage=4082, train=7.2s


planned=0.3, seed=2026: NDCG@10=0.099313, coverage=4097, train=6.9s


planned=0.5, seed=42: NDCG@10=0.100216, coverage=4253, train=7.0s


planned=0.5, seed=7: NDCG@10=0.099340, coverage=4219, train=6.8s


planned=0.5, seed=2026: NDCG@10=0.099730, coverage=4256, train=7.0s


planned=1.0, seed=42: NDCG@10=0.099488, coverage=4569, train=7.3s


planned=1.0, seed=7: NDCG@10=0.098590, coverage=4548, train=6.9s


planned=1.0, seed=2026: NDCG@10=0.097830, coverage=4530, train=6.9s


precision@10  recall@10  ndcg@10  coverage@10  users  \
planned_weight seed                                                         
0.300000       42        0.081417   0.063732 0.099330         4077   7184   
               7         0.080777   0.062588 0.098172         4082   7184   
               2026      0.081389   0.063433 0.099313         4097   7184   
0.500000       42        0.081974   0.064405 0.100216         4253   7184   
               7         0.081584   0.063838 0.099340         4219   7184   
               2026      0.081807   0.064172 0.099730         4256   7184   
1.000000       42        0.081473   0.064769 0.099488         4569   7184   
               7         0.080498   0.063706 0.098590         4548   7184   
               2026      0.080150   0.063406 0.097830         4530   7184   

                     training_rows  short_lists  fit_seconds  \
planned_weight seed                                            
0.300000       42          3293589            0     7.005630   
               7           3293589            0     7.222048   
               2026        3293589            0     6.909273   
0.500000       42          3293589            0     7.040867   
               7           3293589            0     6.754522   
               2026        3293589            0     6.993098   
1.000000       42          3293589            0     7.266156   
               7           3293589            0     6.890254   
               2026        3293589            0     6.934035   

                     recommend_seconds  
planned_weight seed                     
0.300000       42             9.003112  
               7              9.022836  
               2026           8.865086  
0.500000       42             8.870759  
               7              8.878298  
               2026           9.187300  
1.000000       42             9.011672  
               7              9.606438  
               2026           8.806882

## Среднее качество и устойчивость

Std описывает разброс между тремя инициализациями на одном validation.
Выбираем предварительного кандидата по среднему NDCG, но не считаем маленькую
разницу доказательством превосходства и пока не меняем значения по умолчанию.

In [4]:
summary = results.groupby("planned_weight").agg(
    ndcg_mean=("ndcg@10", "mean"), ndcg_std=("ndcg@10", "std"),
    ndcg_min=("ndcg@10", "min"), ndcg_max=("ndcg@10", "max"),
    precision_mean=("precision@10", "mean"), recall_mean=("recall@10", "mean"),
    coverage_mean=("coverage@10", "mean"))
display(summary)
candidate = float(summary.ndcg_mean.idxmax())
print("Лучшее среднее NDCG среди проверенных весов planned:", candidate)
print("NDCG baseline популярности:", popular_metrics["ndcg@10"])
paired = results.pivot(index="seed", columns="planned_weight", values="ndcg@10")
display(paired.subtract(paired[0.5], axis=0).rename_axis(columns="разница с planned=0.5"))
# Парные изменения по пользователям, усреднённые по тем же seed.
delta_rows = []
for planned in [0.3, 1.0]:
    delta = np.mean([per_user[(planned, seed)] - per_user[(0.5, seed)] for seed in SEEDS], axis=0)
    delta_rows.append({"planned_weight": planned, "mean_delta": delta.mean(),
                       "лучше": int((delta > 1e-12).sum()),
                       "без изменения": int((np.abs(delta) <= 1e-12).sum()),
                       "хуже": int((delta < -1e-12).sum())})
user_deltas = pd.DataFrame(delta_rows).set_index("planned_weight")
display(user_deltas)

,ndcg_mean,ndcg_std,ndcg_min,ndcg_max,precision_mean,recall_mean,coverage_mean
planned_weight,,,,,,,
0.300000,0.098939,0.000664,0.098172,0.099330,0.081194,0.063251,"4,085.333333"
0.500000,0.099762,0.000438,0.099340,0.100216,0.081788,0.064138,"4,242.666667"
1.000000,0.098636,0.000830,0.097830,0.099488,0.080707,0.063961,"4,549.000000"


Лучшее среднее NDCG среди проверенных весов planned: 0.5
NDCG baseline популярности: 0.11302101779723082


разница с planned=0.5,0.300000,0.500000,1.000000
seed,,,
7,-0.001168,0.000000,-0.000751
42,-0.000885,0.000000,-0.000728
2026,-0.000417,0.000000,-0.001900


,mean_delta,лучше,без изменения,хуже
planned_weight,,,,
0.300000,-0.000823,1477,4087,1620
1.000000,-0.001126,1667,3832,1685


## Артефакты опыта

Сохраняем таблицы результатов и контекст в `ml/artifacts/als_planned_weight/`
(вне Git). Полные таблицы также остаются в выводах этого ноутбука. Модель
`als_baseline.npz` и глобальные defaults не перезаписываются.

In [5]:
out = Path.cwd().parent / "artifacts" / "als_planned_weight"
out.mkdir(parents=True, exist_ok=True)
results.to_csv(out / "runs.csv", index=False)
summary.to_csv(out / "summary.csv")
paired.subtract(paired[0.5], axis=0).to_csv(out / "paired_deltas.csv")
user_deltas.to_csv(out / "user_deltas.csv")
(out / "protocol.json").write_text(json.dumps({
    "snapshot_end": SNAPSHOT_END.isoformat(), "validation_start": VALIDATION_START.isoformat(),
    "test_start": TEST_START.isoformat(), "test_evaluated": False,
    "planned_weights": PLANNED_WEIGHTS, "score_coef": SCORE_COEF, "seeds": SEEDS, "als_params": ALS_PARAMS,
    "min_user_items": 5, "min_item_users": 5,
    "base_status_weights": BASE_STATUS_WEIGHTS,
    "status_weights_by_planned": {str(p): {**BASE_STATUS_WEIGHTS, "planned": p} for p in PLANNED_WEIGHTS}, "min_score": 5, "min_user_scores": 5,
    "clip": [0.5, 1.5], "half_life_days": None,
    "user_id_namespace": "anilist", "item_id_namespace": "anilist",
    "warm_validation_users": len(users), "confidence": "1 + alpha * weight",
    "versions": versions, "popular_baseline": popular_metrics,
}, ensure_ascii=False, indent=2) + "\n")
print("Сохранены:", ", ".join(p.name for p in sorted(out.iterdir())))

Сохранены: paired_deltas.csv, protocol.json, runs.csv, summary.csv, user_deltas.csv


## Вывод

| planned | Средний NDCG@10 | Std между seed | Precision@10 | Recall@10 | Среднее coverage@10 |
|---|---:|---:|---:|---:|---:|
| 0,3 | 0,098939 | 0,000664 | 0,081194 | 0,063251 | 4085,3 |
| 0,5 | 0,099762 | 0,000438 | 0,081788 | 0,064138 | 4242,7 |
| 1,0 | 0,098636 | 0,000830 | 0,080707 | 0,063961 | 4549,0 |

- **Оставляем planned=0,5** для следующего последовательного опыта при
  score_coef=0. Это лучший средний NDCG среди проверенных вариантов; вес 0,5
  лучше обоих альтернативных весов на каждом из трёх seed.
- При снижении веса до 0,3 средний NDCG уменьшается на 0,000823 (~0,83%),
  при повышении до 1,0 — на 0,001126 (~1,13%). Разницы небольшие; три seed
  на одном validation не доказывают статистическую значимость или переносимость.
- Вес 1,0 увеличивает покрытие с 4242,7 до 4549 тайтлов (~7,2%), но ухудшает
  средний NDCG, precision и recall. Вес 0,3 снижает все четыре средние метрики.
  Покрытие — количество уникальных тайтлов, здесь усреднённое по seed.
- При planned=0,5 все метрики на каждом seed точно воспроизвели score_coef=0
  из ноутбука 04. Во всех девяти запусках одинаковые 3 293 589 положительных пар
  и полный топ-10 для 7 184 пользователей, без дублей и пересечений с train-history.
- Популярность за 28 дней: NDCG@10 **0,113021**, все варианты ALS ниже.
  Изменение planned не закрывает разрыв с контрольным baseline.
- Следующий опыт — порог 5/5 против 5/20 при score_coef=0 и planned=0,5,
  с прежними временными границами и неизменными целями warm-validation.
  Затем можно проверить настройки ALS. Defaults и исходная модель не изменены;
  test при подборе параметров не оценивался.
